<div align="left" style="background-color: #008080; padding: 20px 10px;">
<h3><b>IDEAS - Institute of Data Engineering, Analytics and Science Foundation</b></h3>
<p>Summer Internship Program 2026</p>
<hr style="width:100%;">
<h3><b>Project Title:</b> Exploratory Data Analysis with Sales Data</h3>
<h4>Project Notebook</h4>

<blockquote style="border-left: 4px solid #4285F4; padding-left: 15px;">
  <strong>Created by:</strong> Sanchaita Chakraborty<br>
  <strong>Designation:</strong> Project Linked Associate Research Engineer<br>
</blockquote>
<hr style="width:100%;">
</div>

### Question 1: Import Data and Store-wise Revenue Analysis (4 Marks)

Import `pandas` and `numpy`. Download the dataset `sales.csv` from https://drive.google.com/drive/folders/1gieHICVDBbUKMZiSF4YRQMAJic-w50JM?usp=sharing. Load the dataset `sales.csv` into a DataFrame. Then, find the top 5 stores with the highest total sales and the bottom 5 stores with the lowest total sales.

**Tasks:**
- Aggregate total sales by store_id
- Rank stores by total revenue
- Compare average sales per department within those stores

In [28]:
import pandas as pd
import numpy as np
df=pd.read_csv("sales.csv") #reading the csv file


#aggregating weekly sales on the basis of store id
total=df.groupby("store_id")["weekly_sales"].sum().reset_index()
total.columns=["store_id","total sales"]
print(total.to_string(index=False))

#ranking stores by total revenue
total=total.sort_values("total sales",ascending=False).reset_index(drop=True)
total["rank"]=total["total sales"].rank(ascending=False,method="dense").astype(int)
top5=total.head(5)
bottom5=total.tail(5)
print("Top 5 stores:")
print(top5.to_string(index=False))
print("\nBottom 5 stores:")
print(bottom5.to_string(index=False))


# Comparing Average Sales per Department within Top 5 & Bottom 5
top5_lst=top5["store_id"].tolist()
bottom5_lst=bottom5["store_id"].tolist()
top5_avg=(df[df["store_id"].isin(top5_lst)]
.groupby(["store_id","department"])["weekly_sales"].mean()
.reset_index().rename(columns={"weekly_sales":"top5_avg"})
.sort_values(["store_id","top5_avg"],ascending=[True,False]))
bottom5_avg=(df[df["store_id"].isin(bottom5_lst)]
.groupby(["store_id","department"])["weekly_sales"].mean()
.reset_index().rename(columns={"weekly_sales":"bottom5_avg"})
.sort_values(["store_id","bottom5_avg"],ascending=[True,False]))
print("Average Sales per Department within Top 5:")
print(top5_avg.to_string(index=False))
print("\nAverage Sales per Department within Bottom 5:")
print(bottom5_avg.to_string(index=False))

 store_id  total sales
        1 299904995.68
        2  27377684.86
        3 115529958.18
        4  96965353.15
        5 300063876.99
        6 290117938.85
        7 186728203.44
        8  31361162.23
        9 112887361.55
       10 159792226.62
       11 292354024.81
       12  38613465.75
       13  60273453.26
       14 184862680.99
       15 258960009.33
       16 235567027.76
       17 218793108.70
       18  92781541.07
       19 253132540.53
       20 245499350.98
       21 166653393.63
       22 188843741.03
       23 279639036.60
       24 253447308.44
       25 160251436.91
       26 130805423.94
       27 192448691.95
       28 111153406.08
       29  90800226.84
       30 202004512.94
       31 118688120.71
       32 303223049.45
       33 208793741.73
       34  46033475.43
       35  64256626.86
       36  65444402.71
       37 291374338.78
       38 280749382.09
       39 103191846.38
       40 253911559.72
       41 301811851.20
       42 124895294.49
       43 2

### Question 2: Department Performance Consistency (3 Marks)

Which departments have the most stable weekly sales and which are the most volatile?

**Tasks:**
- Use standard deviation and coefficient of variation
- Identify highly stable and highly volatile departments

In [29]:
#  Standard Deviation & Coefficient of Variation

dept_stats = df.groupby('department')['weekly_sales'].agg(
    mean_sales='mean',
    std_sales='std'
).reset_index()

dept_stats['cv'] = (dept_stats['std_sales'] / dept_stats['mean_sales']) * 100

print("=== Department Performance Consistency ===")
print(dept_stats.to_string(index=False))

# Identifying Stable and Volatile Departments

dept_stats_sorted = dept_stats.sort_values('cv', ascending=True).reset_index(drop=True)

most_stable   = dept_stats_sorted.head(3)
most_volatile = dept_stats_sorted.tail(3)

print("\n=== Most Stable Departments (Low CV) ===")
print(most_stable.to_string(index=False))

print("\n=== Most Volatile Departments (High CV) ===")
print(most_volatile.to_string(index=False))


=== Department Performance Consistency ===
 department   mean_sales    std_sales        cv
          1 37200.611536 28781.821438 77.369216
          2 39641.066051 30757.242670 77.589343
          3 41665.968603 32750.695053 78.602985
          4 43276.523323 33851.231686 78.220774
          5 44963.078123 35413.433164 78.761141
          6 47729.393928 37045.408507 77.615502
          7 48995.418363 38407.713335 78.390418
          8 51600.325251 40549.772807 78.584336
          9 53781.526778 42396.192677 78.830400
         10 55648.465719 43399.339844 77.988385
         11 58219.685860 45305.657547 77.818451
         12 59896.227024 46869.174762 78.250630
         13 60887.532814 46971.400142 77.144529
         14 63046.177503 48940.888590 77.627051
         15 65246.817618 51315.229169 78.647865
         16 67908.462991 53568.802692 78.883839
         17 69205.177045 55319.077732 79.934884
         18 71819.047794 56492.667557 78.659728
         19 73401.916464 57421.490851 78.2288

### Question 3: Holiday vs Non-Holiday Sales (3 Marks)

Analyze whether holidays significantly affect weekly sales.

**Tasks:**
- Compare average sales during holidays vs non-holidays
- Calculate percentage increase/decrease
- Identify departments benefiting most from holidays

In [30]:
# Comparing Average Sales During Holidays vs Non-Holidays

holiday_avg    = df[df['is_holiday'] == 1]['weekly_sales'].mean()
nonholiday_avg = df[df['is_holiday'] == 0]['weekly_sales'].mean()

print("=== Holiday vs Non-Holiday Average Sales ===")
print(f"Holiday Average Sales     : {holiday_avg:.2f}")
print(f"Non-Holiday Average Sales : {nonholiday_avg:.2f}")


# Calculating Percentage Increase/Decrease

pct_change = ((holiday_avg - nonholiday_avg) / nonholiday_avg) * 100

print("\n=== Percentage Change ===")
if pct_change > 0:
    print(f"Holidays INCREASE sales by : {pct_change:.2f}%")
else:
    print(f"Holidays DECREASE sales by : {abs(pct_change):.2f}%")

# Identifying Departments Benefiting Most from Holidays

dept_holiday    = df[df['is_holiday'] == True].groupby('department')['weekly_sales'].mean()
dept_nonholiday = df[df['is_holiday'] == False].groupby('department')['weekly_sales'].mean()

dept_comparison = pd.DataFrame({
    'holiday_avg'    : dept_holiday,
    'nonholiday_avg' : dept_nonholiday
}).reset_index()

dept_comparison['pct_change'] = (
    (dept_comparison['holiday_avg'] - dept_comparison['nonholiday_avg'])
    / dept_comparison['nonholiday_avg']
) * 100

dept_comparison = dept_comparison.sort_values('pct_change', ascending=False).reset_index(drop=True)

print("\n=== Department-wise Holiday Impact ===")
print(dept_comparison.to_string(index=False))

print("\n=== Top 3 Departments Benefiting Most from Holidays ===")
print(dept_comparison.head(3).to_string(index=False))

print("\n=== Top 3 Departments Least Affected by Holidays ===")
print(dept_comparison.tail(3).to_string(index=False))



=== Holiday vs Non-Holiday Average Sales ===
Holiday Average Sales     : 84493.83
Non-Holiday Average Sales : 52852.58

=== Percentage Change ===
Holidays INCREASE sales by : 59.87%

=== Department-wise Holiday Impact ===
 department   holiday_avg  nonholiday_avg  pct_change
         17 106740.490344    64309.266614   65.979953
          7  75408.944344    45550.175843   65.551379
          9  82456.372178    50041.329552   64.776542
         11  88058.513233    54327.664899   62.087793
          3  62865.730489    38900.782270   61.605312
          5  67745.178900    41991.499761   61.330696
         16 102204.816311    63435.025601   61.117325
          8  77445.604822    48229.201829   60.578243
         20 113946.098278    70978.560422   60.535939
         18 107698.553044    67139.112326   60.411047
         19 109712.451956    68665.759661   59.777526
          1  55407.872811    34825.751370   59.100294
          2  58871.147589    37132.794546   58.542195
          4  64096.268

### Question 4: Seasonal Trend Detection (3 Marks)

Identify whether sales increase or decrease during specific months.

**Tasks:**
- Extract month from the date column
- Compute average monthly sales
- Determine highest and lowest sales months

In [31]:
# Extracting Month from Date Column

df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()

print("=== Date Column After Extraction ===")
print(df[['date', 'month', 'month_name', 'weekly_sales']].head(10).to_string(index=False))

# Computing Average Monthly Sales

monthly_avg = df.groupby(['month', 'month_name'])['weekly_sales'].mean().reset_index()
monthly_avg.columns = ['month', 'month_name', 'avg_sales']
monthly_avg = monthly_avg.sort_values('month').reset_index(drop=True)
monthly_avg['avg_sales'] = monthly_avg['avg_sales'].round(2)

print("\n=== Average Monthly Sales ===")
print(monthly_avg.to_string(index=False))

# Determining Highest and Lowest Sales Months

highest_month = monthly_avg.loc[monthly_avg['avg_sales'].idxmax()]
lowest_month  = monthly_avg.loc[monthly_avg['avg_sales'].idxmin()]

print("\n=== Highest Sales Month ===")
print(f"Month    : {highest_month['month_name']}")
print(f"Avg Sales: {highest_month['avg_sales']:.2f}")

print("\n=== Lowest Sales Month ===")
print(f"Month    : {lowest_month['month_name']}")
print(f"Avg Sales: {lowest_month['avg_sales']:.2f}")


=== Date Column After Extraction ===
      date  month month_name  weekly_sales
2022-01-01      1    January     119075.96
2022-01-01      1    January     119107.85
2022-01-01      1    January      84369.88
2022-01-01      1    January      88445.24
2022-01-01      1    January      65159.85
2022-01-01      1    January      99015.94
2022-01-01      1    January     157858.77
2022-01-01      1    January      58702.96
2022-01-01      1    January     199677.02
2022-01-01      1    January     139744.34

=== Average Monthly Sales ===
 month month_name  avg_sales
     1    January   59409.68
     2   February   56327.29
     3      March   49096.70
     4      April   49509.00
     5        May   48642.59
     6       June   49130.17
     7       July   51878.72
     8     August   49289.19
     9  September   62113.26
    10    October   58702.35
    11   November   73301.88
    12   December   70916.60

=== Highest Sales Month ===
Month    : November
Avg Sales: 73301.88

=== Lowest S

### Question 5: Store and Department Contribution (4 Marks)

Which combinations of store and department contribute the most to overall revenue?

**Tasks:**
- Group using store_id and department
- Find top and worst-performing combinations
- Compute percentage contribution

In [32]:
# Grouping by store_id and department

combo_sales = df.groupby(['store_id', 'department'])['weekly_sales'].sum().reset_index()
combo_sales.columns = ['store_id', 'department', 'total_sales']

print("=== Store & Department Combinations ===")
print(combo_sales.head(6).to_string(index=False))

# Finding Top and Worst Performing Combinations

combo_sales = combo_sales.sort_values('total_sales', ascending=False).reset_index(drop=True)

top5_combo    = combo_sales.head(5)
bottom5_combo = combo_sales.tail(5)

print("\n=== Top 5 Best Performing Combinations ===")
print(top5_combo.to_string(index=False))

print("\n=== Bottom 5 Worst Performing Combinations ===")
print(bottom5_combo.to_string(index=False))
# Computing Percentage Contribution

total_revenue = combo_sales['total_sales'].sum()

combo_sales['pct_contribution'] = (combo_sales['total_sales'] / total_revenue) * 100
combo_sales['pct_contribution'] = combo_sales['pct_contribution'].round(4)

print("\n=== All Combinations with % Contribution ===")
print(combo_sales.to_string(index=False))

print("\n=== Top 5 Combinations by % Contribution ===")
print(combo_sales.head(5).to_string(index=False))

print("\n=== Bottom 5 Combinations by % Contribution ===")
print(combo_sales.tail(5).to_string(index=False))

print(f"\nTotal Revenue (all stores + departments) : {total_revenue:,.2f}")
print(f"Top 5 Contribution Sum                   : {combo_sales.head(5)['pct_contribution'].sum():.2f}%")



=== Store & Department Combinations ===
 store_id  department  total_sales
        1           1   9676831.18
        1           2  10228070.49
        1           3  10865215.71
        1           4  11932582.57
        1           5  11858276.09
        1           6  12677372.69

=== Top 5 Best Performing Combinations ===
 store_id  department  total_sales
        5          20  21949526.84
       32          19  21348232.29
        1          20  20751644.53
       41          20  20744989.41
       32          20  20494554.56

=== Bottom 5 Worst Performing Combinations ===
 store_id  department  total_sales
        2           4   1060057.16
        8           1   1045511.61
        8           2   1037592.33
        2           2    929603.27
        2           1    903803.12

=== All Combinations with % Contribution ===
 store_id  department  total_sales  pct_contribution
        5          20  21949526.84            0.2490
       32          19  21348232.29            0.242

### Question 6: Load MNIST Data (3 Marks)

Load the MNIST dataset using `sklearn.datasets.load_digits`. Separate the dataset into features (`X`) and target labels (`y`).
Print the shape of the features and the target arrays.

**Expected Output:** The shape of `X` and `y`.

In [33]:
from sklearn.datasets import load_digits

# Loading the MNIST digits dataset
digits = load_digits()

# Separating into features and target labels
X = digits.data
y = digits.target

# Printing the shapes
print(f"Shape of X (features) : {X.shape}")
print(f"Shape of y (target)   : {y.shape}")

Shape of X (features) : (1797, 64)
Shape of y (target)   : (1797,)


### Question 7: K-Means Clustering (5 Marks)

Import `KMeans` from `sklearn.cluster`. Initialize and fit the K-Means clustering model on the MNIST features (`X`).
Since we know there are 10 digits (0-9), set the number of clusters to 10. Assign the cluster labels to a variable `kmeans_labels`.

**Expected Output:** A successfully fitted K-Means model and the cluster labels array.

In [34]:
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans

# Loading the dataset
digits = load_digits()
X = digits.data
y = digits.target

print(f"Shape of X : {X.shape}")

# Initialising the KMeans model
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)

# Fitting the model on the features
kmeans.fit(X)

# Assigning cluster labels
kmeans_labels = kmeans.labels_

# Printing results
print(f"Cluster Labels : {kmeans_labels}")
print(f"Shape of kmeans_labels : {kmeans_labels.shape}")
print(f"Unique Clusters : {set(kmeans_labels)}")

Shape of X : (1797, 64)
Cluster Labels : [1 3 3 ... 3 8 8]
Shape of kmeans_labels : (1797,)
Unique Clusters : {np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9)}


### Question 8: F1 Score Evaluation for Clustering (5 Marks)

Evaluate the clustering performance using the F1 score. Since K-Means assigns arbitrary cluster labels, you will first need to map each cluster label to the most frequent true label in that cluster.
After mapping the labels, calculate and print the macro-averaged F1 score using `sklearn.metrics.f1_score`.

**Expected Output:** The calculated F1 score of the clustering.

In [35]:
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score
import numpy as np

# Loading Data
digits = load_digits()
X = digits.data
y = digits.target

# Fitting KMeans
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
kmeans.fit(X)
kmeans_labels = kmeans.labels_

#  Mapping Each Cluster Label to Most Frequent True Label
label_mapping = {}

for cluster in range(10):
    mask = (kmeans_labels == cluster)
    true_labels_in_cluster = y[mask]
    most_frequent = np.bincount(true_labels_in_cluster).argmax()
    label_mapping[cluster] = most_frequent

print(f"Cluster → Digit Mapping : {label_mapping}")

# Applying the Mapping
mapped_labels = np.array([label_mapping[label] for label in kmeans_labels])

print(f"True Labels   : {y[:10]}")
print(f"Mapped Labels : {mapped_labels[:10]}")

#  Calculating Macro-Averaged F1 Score
f1 = f1_score(y, mapped_labels, average='macro')

print(f"\nMacro-Averaged F1 Score : {f1:.4f}")

Cluster → Digit Mapping : {0: np.int64(2), 1: np.int64(0), 2: np.int64(1), 3: np.int64(8), 4: np.int64(7), 5: np.int64(6), 6: np.int64(3), 7: np.int64(5), 8: np.int64(9), 9: np.int64(4)}
True Labels   : [0 1 2 3 4 5 6 7 8 9]
Mapped Labels : [0 8 8 3 4 9 6 7 8 9]

Macro-Averaged F1 Score : 0.7895
